# Audience Seeding · Step 3 · Colab GPU jobs (free)

Three jobs that need a GPU. Each is judged later by the same backtest and keep/drop rule as everything else,
and is kept only if it measurably improves the forecasts.

| section | what | model | time on a free T4 | output |
|---|---|---|---|---|
| A | text model: how much a post beats its author's usual, from the words | `BAAI/bge-small-en-v1.5` (fine-tuned) | ~30–45 min | `text_ft_preds.parquet` |
| B | content grader: hook, specificity, novelty… for every post | `Qwen/Qwen2.5-3B-Instruct` (free, Apache-2.0) | ~45–90 min | `llm_grades.parquet` |
| C | audience panel: how real people from each client's audience would react | `Qwen/Qwen2.5-7B-Instruct-AWQ` | ~60–120 min | `panel_scores.parquet` |

**Before you start**
1. Have these 3 files ready from `D:\Audience_seeding\data\colab\`: `train_posts.parquet`, `panel_prompts.parquet`, `folds.json`. **Setup** shows an upload button: choose all three at once.
2. Runtime → Change runtime type → **T4 GPU**.
3. Run **Setup**, then one section at a time. Each section **downloads its result to your PC when it finishes**.
   If Colab disconnects, uploaded files are lost: run Setup again (re-upload) and re-run the section.
   (Optional: set `USE_DRIVE = True` to keep files and progress in Google Drive instead.)
4. **Restart the runtime between sections B and C** (Runtime → Restart session), then run Setup again.
   vLLM keeps GPU memory until a restart.
5. When done, send back the three downloaded `.parquet` files.

## Setup (run first, and again after any restart)

In [ ]:
USE_DRIVE = False   # False: upload the files by hand below. True: read/save from Google Drive folder 'audience_seeding'
import os, torch
NEEDED = ['train_posts.parquet', 'panel_prompts.parquet', 'folds.json']
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    D = '/content/drive/MyDrive/audience_seeding'
else:
    D = '/content/audience_seeding'
os.makedirs(f'{D}/parts', exist_ok=True)
missing = [f for f in NEEDED if not os.path.exists(f'{D}/{f}')]
if missing and not USE_DRIVE:
    from google.colab import files
    print('Choose these files (from the data/colab folder of the project):', missing)
    for name, data in files.upload().items():
        open(f'{D}/{name}', 'wb').write(data)
print('files:', sorted(os.listdir(D)))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set the runtime to T4 GPU')

def send_back(path):
    """Download a finished output to your PC (it is also kept in the folder above)."""
    if not USE_DRIVE:
        from google.colab import files
        files.download(path)

In [ ]:
import json, re
import numpy as np, pandas as pd

GRADE_FIELDS = ['hook', 'specificity', 'novelty', 'contrarian', 'emotion', 'practical_value', 'cta', 'bait', 'negative_risk', 'reply_invite']
GRADE_SYSTEM = 'You grade X (Twitter) posts for a content team. Score the post text only, each 0-10 (integers):\nhook: how strongly the first line makes a scrolling reader stop.\nspecificity: concrete numbers, names, steps, examples (10) vs vague generalities (0).\nnovelty: a fresh idea or angle for its audience (10) vs a recycled take (0).\ncontrarian: goes against common belief in its niche.\nemotion: emotional intensity of any kind.\npractical_value: a reader can act on it today.\ncta: strength of any call to action (comment, repost, follow, link, DM).\nbait: engagement bait (comment-to-get, giveaways, "like if", rage bait, follow-for-follow).\nnegative_risk: how likely typical readers are to hit "not interested", mute, block or report (spammy, offensive,\n  misleading, off-putting self-promotion).\nreply_invite: how naturally it invites replies (open question, debate, ask for opinions).\nJudge the text as written. Do not guess the author\'s popularity.' + "\n\nReply with ONLY a JSON object with these integer keys (0-10): " + ", ".join(GRADE_FIELDS) + "."
PANEL_JSON = ('\n\nReply with ONLY a JSON object: {"people": [{"handle": "...", "action": "scroll|like|reply|repost|quote", '
              '"p_engage": 0.0-1.0, "p_reply": 0.0-1.0, "reply": "..."}], "objections": ["..."], "who_reacts": "..."}. '
              'Include every person listed, once.')

def _json_obj(text):
    """First {...} block in a model answer, parsed; None if there isn't a valid one."""
    m = re.search(r"\{.*\}", text or "", re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def parse_grade(text):
    d = _json_obj(text)
    if not isinstance(d, dict):
        return None
    out = {}
    for f in GRADE_FIELDS:
        try:
            out[f] = int(max(0, min(10, round(float(d[f])))))
        except (KeyError, TypeError, ValueError):
            return None
    return out

def parse_panel(text, people):
    """Model answer + the panel's people (list of {handle, segment, kind}) -> one summary row, or None."""
    d = _json_obj(text)
    if not isinstance(d, dict) or not isinstance(d.get("people"), list):
        return None
    seg = {p["handle"].lower(): (p.get("segment") or "unknown") for p in people}
    rows = []
    for p in d["people"]:
        if not isinstance(p, dict):
            continue
        h = str(p.get("handle", "")).lstrip("@").lower()
        if h not in seg:
            continue
        try:
            pe, pr = float(p.get("p_engage", 0)), float(p.get("p_reply", 0))
        except (TypeError, ValueError):
            continue
        rows.append({"handle": h, "segment": seg[h], "p_engage": min(max(pe, 0), 1), "p_reply": min(max(pr, 0), 1),
                     "reply": str(p.get("reply", ""))[:200]})
    if len(rows) < max(3, len(people) // 2):          # the model skipped most people: don't trust it
        return None
    r = pd.DataFrame(rows).drop_duplicates("handle")
    by = r.groupby("segment").p_engage.mean().round(3).to_dict()
    top = r[r.reply.str.len() > 0].sort_values("p_reply", ascending=False).head(3)
    return {"panel_n": len(r), "panel_engage": float(r.p_engage.mean()), "panel_reply": float(r.p_reply.mean()),
            "panel_by_niche": json.dumps(by), "panel_replies": json.dumps(top[["handle", "reply"]].to_dict("records")),
            "panel_objections": json.dumps([str(x)[:200] for x in d.get("objections", [])][:3])}

## A · Fine-tuned text model
Each fold's model trains **only** on that fold's `train` posts and predicts its `cal` and `test` posts (folds are fixed
in the file, so it can't peek at test posts). Saves one part per fold × target.

In [ ]:
!pip -q install "transformers>=4.44" datasets accelerate

In [ ]:
import pandas as pd, numpy as np, torch, os
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          DataCollatorWithPadding)
df = pd.read_parquet(f'{D}/train_posts.parquet')
MODEL = 'BAAI/bge-small-en-v1.5'
tok = AutoTokenizer.from_pretrained(MODEL)

def ds(frame, label=None):
    d = {'text': frame.text.fillna('').str.slice(0, 1000).tolist()}
    if label is not None:
        d['labels'] = label.astype('float32').tolist()
    return Dataset.from_dict(d).map(lambda b: tok(b['text'], truncation=True, max_length=128), batched=True,
                                    remove_columns=['text'])

for k in range(1, 5):
    for t in ['views', 'xscore']:
        part = f'{D}/parts/textft_f{k}_{t}.parquet'
        if os.path.exists(part):
            print('done already:', part); continue
        f = f'fold{k}'
        tr = df[(df[f] == 'train')].dropna(subset=[f'y_{t}', f'lb_{t}'])
        ev_ = df[df[f].isin(['cal', 'test'])]
        resid = (tr[f'y_{t}'] - tr[f'lb_{t}']).to_numpy(); mu, sd = resid.mean(), resid.std()
        model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1, problem_type='regression')
        args = TrainingArguments(output_dir='/tmp/ft', per_device_train_batch_size=64, num_train_epochs=2,
                                 learning_rate=3e-5, warmup_ratio=0.06, weight_decay=0.01, fp16=True,
                                 logging_steps=200, save_strategy='no', report_to=[], seed=0)
        Trainer(model=model, args=args, train_dataset=ds(tr, (resid - mu) / sd),
                data_collator=DataCollatorWithPadding(tok)).train()
        pred = Trainer(model=model, args=args, data_collator=DataCollatorWithPadding(tok)).predict(ds(ev_))
        pd.DataFrame({'fold': k, 'tweet_id': ev_.tweet_id.to_numpy(), 'target': t, 'split': ev_[f].to_numpy(),
                      'ft_pred': pred.predictions.ravel() * sd + mu}).to_parquet(part, index=False)
        print('saved', part, len(ev_)); del model; torch.cuda.empty_cache()
parts = [p for p in os.listdir(f'{D}/parts') if p.startswith('textft_')]
pd.concat([pd.read_parquet(f'{D}/parts/{p}') for p in parts]).to_parquet(f'{D}/text_ft_preds.parquet', index=False)
print('A done ->', f'{D}/text_ft_preds.parquet', len(parts), 'parts')
send_back(f'{D}/text_ft_preds.parquet')

## B · Free content grader (Qwen2.5-3B-Instruct on vLLM)
Grades every post on the same rubric the pipeline uses. Saves every 2,000 posts. If `import vllm` fails after the
install, do Runtime → Restart session, run Setup + the helpers cell again, then this section again.

In [ ]:
!pip -q install vllm

In [ ]:
import pandas as pd, os
from vllm import LLM, SamplingParams
posts = pd.read_parquet(f'{D}/train_posts.parquet')[['tweet_id', 'text']].drop_duplicates('tweet_id')
llm = LLM(model='Qwen/Qwen2.5-3B-Instruct', dtype='half', max_model_len=2048, gpu_memory_utilization=0.90,
          enable_prefix_caching=True)
sp = SamplingParams(temperature=0.0, max_tokens=120)
CH = 2000
for i in range(0, len(posts), CH):
    part = f'{D}/parts/grades_{i // CH:03d}.parquet'
    if os.path.exists(part):
        continue
    chunk = posts.iloc[i:i + CH]
    msgs = [[{'role': 'system', 'content': GRADE_SYSTEM},
             {'role': 'user', 'content': 'Post:\n' + str(t or '')[:2500]}] for t in chunk.text]
    outs = llm.chat(msgs, sp, use_tqdm=True)
    rows = []
    for tid, o in zip(chunk.tweet_id, outs):
        g = parse_grade(o.outputs[0].text)
        if g:
            rows.append({'tweet_id': tid, **g})
    pd.DataFrame(rows).to_parquet(part, index=False)
    print(f'saved {part}: {len(rows)}/{len(chunk)} parsed')
parts = sorted(p for p in os.listdir(f'{D}/parts') if p.startswith('grades_'))
g = pd.concat([pd.read_parquet(f'{D}/parts/{p}') for p in parts])
g['model'] = 'Qwen2.5-3B-Instruct'
g.to_parquet(f'{D}/llm_grades.parquet', index=False)
print('B done ->', len(g), 'posts graded')
send_back(f'{D}/llm_grades.parquet')

## C · Free audience panel (Qwen2.5-7B-Instruct-AWQ on vLLM)
**Restart the runtime first** (Runtime → Restart session), then run Setup and the helpers cell.
Each prompt lists ~20 real people from the client's audience (their bios and posts, only from before the post).
The model says how each would react. Saves every 40 posts.

In [ ]:
import pandas as pd, os, json
from vllm import LLM, SamplingParams
P = pd.read_parquet(f'{D}/panel_prompts.parquet')
llm = LLM(model='Qwen/Qwen2.5-7B-Instruct-AWQ', quantization='awq', dtype='half', max_model_len=12288,
          gpu_memory_utilization=0.92)
sp = SamplingParams(temperature=0.2, max_tokens=2500)
CH = 40
for i in range(0, len(P), CH):
    part = f'{D}/parts/panel_{i // CH:03d}.parquet'
    if os.path.exists(part):
        continue
    chunk = P.iloc[i:i + CH]
    msgs = [[{'role': 'system', 'content': s + PANEL_JSON}, {'role': 'user', 'content': u[:40000]}]
            for s, u in zip(chunk.system, chunk.prompt)]
    outs = llm.chat(msgs, sp, use_tqdm=True)
    rows = []
    for r, o in zip(chunk.itertuples(), outs):
        s = parse_panel(o.outputs[0].text, json.loads(r.people))
        if s:
            rows.append({'tweet_id': r.tweet_id, 'client': r.client, **s})
    pd.DataFrame(rows).to_parquet(part, index=False)
    print(f'saved {part}: {len(rows)}/{len(chunk)} parsed')
parts = sorted(p for p in os.listdir(f'{D}/parts') if p.startswith('panel_'))
s = pd.concat([pd.read_parquet(f'{D}/parts/{p}') for p in parts])
s['model'] = 'Qwen2.5-7B-Instruct-AWQ'
s.to_parquet(f'{D}/panel_scores.parquet', index=False)
print('C done ->', len(s), 'posts with a panel')
send_back(f'{D}/panel_scores.parquet')

## Send back
Each section downloaded its file when it finished. Send these three:
`text_ft_preds.parquet`, `llm_grades.parquet`, `panel_scores.parquet`.